# VAE 하이퍼파라미터 순차 튜닝

`02_vae_mnist.ipynb`의 ResNet VAE를 그대로 사용하여 **β → 잠재 차원 → 학습률 → 모델 용량 → 배치 크기 → KL annealing** 순으로 비교합니다. 각 단계에서는 직전 단계의 선택값을 고정하고 한 변수만 변경합니다. 각 후보는 동일한 seed로 처음부터 학습하며, 이미 실행한 같은 설정은 결과를 재사용합니다.

이 실험은 모든 조합의 최적값을 찾는 전수 탐색이 아닙니다. 변수 간 상호작용과 seed별 변동은 후속 실험으로 확인해야 합니다. 테스트 데이터는 후보 선택에 사용하지 않습니다.

## 1. 비교 조건과 지표

| 항목 | 후보 |
|---|---|
| β | 0.1, 0.5, 1.0, 2.0 |
| 잠재 차원 | 8, 16, 32, 64; 기존 2차원도 대조군으로 유지 |
| Adam 학습률 | 1e-3, 3e-4, 1e-4 |
| 모델 용량 | hidden dimension 400, 800; 채널 32→64→128은 고정 |
| 학습 배치 크기 | 32, 64, 128 |
| KL annealing | 고정 β 또는 전반 50% epoch 동안 0→선택된 β |

**선택 기준**: 모든 후보의 검증 `BCE + KL`(KL 가중치 1)을 동일하게 비교합니다. 학습 total은 `reconstruction + effective_beta * KL`이며 별도로 기록합니다. β가 다른 학습 total을 직접 순위에 사용하면 β 자체가 낮아서 생긴 차이를 개선으로 오해할 수 있습니다. 복원만 최적화하는 기준과 선택 결과가 다를 수 있으므로 reconstruction, KL, pixel MSE와 복원 이미지를 함께 확인합니다.

동일한 학습/검증 인덱스, 검증 배치 크기 128, 검증 샘플링 seed를 사용합니다. 배치 크기 변경 시 epoch 수는 같지만 optimizer 업데이트 횟수는 달라집니다. 소규모 탐색의 기본 예산은 각 후보 10 epoch, 학습 6,000장, 검증 1,000장입니다.

MNIST에는 **BCE soft target**을 유지합니다. 실행 모듈은 연속 픽셀용 고정 분산 Gaussian NLL도 지원하지만, 우도까지 동시에 바꾸지 않습니다. Gaussian 실험 시 `likelihood='gaussian'`, `gaussian_sigma=0.1`로 한 변수만 바꾸고 공통 pixel MSE/BCE도 함께 비교하세요. BCE와 Gaussian NLL의 원시 숫자는 서로 같은 척도가 아닙니다.

In [ ]:
# 실행 위치가 프로젝트 루트이든 notebooks이든 프로젝트 경로를 찾음
from pathlib import Path
# 저장된 실험 설정과 결과를 읽기 위한 모듈을 가져옴
import json
# CSV 표를 읽기 위한 모듈을 가져옴
import csv
# 실행 모듈을 프로젝트에서 불러오도록 경로를 추가할 때 사용함
import sys
# 결과 이미지와 손실 곡선을 표시함
import matplotlib.pyplot as plt
# 노트북에 표와 설명을 표시함
from IPython.display import display, Markdown
# PyTorch의 CPU 스레드 수를 설정함
import torch
# 프로젝트 루트를 설정함
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
# 프로젝트 모듈을 import할 수 있도록 검색 경로에 추가함
sys.path.insert(0, str(PROJECT_DIR))
# 비교 실험 함수와 설정 클래스를 가져옴
from examples.vae_tuning import run_search, Config
# 기존 실행 결과를 읽을 디렉토리를 지정함
RESULT_DIR = PROJECT_DIR / 'outputs/notebooks/vae/tuning_20260929'
# True로 바꾸면 새 탐색을 실행함; 이미 존재하는 경로는 덮어쓰지 않음
RUN_SEARCH = False
# 반복 실험에서 불필요한 CPU 스레드 경쟁을 줄임
torch.set_num_threads(2)
# 새 실험이 명시적으로 선택됐을 때만 학습을 실행함
if RUN_SEARCH:
    # 기존 결과가 있으면 RESULT_DIR을 새 경로로 변경한 뒤 실행해야 함
    run_search(RESULT_DIR, epochs=10, train_limit=6000, val_limit=1000, seed=42)
# 결과가 없는 경우 필요한 실행 방법을 안내함
if not (RESULT_DIR / 'summary.csv').exists():
    # 위 설정을 바꿔 학습하거나 README의 CLI 명령으로 실험 결과를 생성함
    raise FileNotFoundError('실험 결과가 없습니다. RUN_SEARCH=True와 새 RESULT_DIR로 실행하세요.')

## 2. 모든 실험 결과 비교

각 행은 검증 공통 점수가 가장 좋았던 epoch입니다. Annealing은 목표 β에 도달한 epoch들 중에서 고릅니다. `summary.csv`에는 파라미터 수, 시간, 최대 GPU 텐서 메모리도 기록됩니다. 시간과 메모리는 현재 환경에서의 관측값입니다.

In [ ]:
# 전체 실험별 최적 epoch 결과를 읽음
with (RESULT_DIR / 'summary.csv').open() as stream:
    # CSV 문자열을 딕셔너리 목록으로 변환함
    rows = list(csv.DictReader(stream))
# 각 단계에서 선택된 후보와 비교 대상 목록을 읽음
selections = json.loads((RESULT_DIR / 'selections.json').read_text())
# 마지막 단계까지 완료됐을 때의 최종 선택값을 읽음
winner = json.loads((RESULT_DIR / 'winner.json').read_text())
# 사용자에게 비교할 핵심 지표의 표 머리글을 구성함
table = '| trial | β | dz | lr | hidden | batch | anneal | epoch | val total | val recon | val KL | BCE+KL | pixel MSE |\n|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|'
# 모든 후보를 같은 형식으로 표시함
for row in rows:
    # 재구성과 KL은 total과 분리하여 표시함
    table += '\n| ' + ' | '.join(str(row[key]) if key in ('trial', 'epoch') else f'{float(row[key]):.5g}' for key in ('trial', 'beta', 'latent_dim', 'learning_rate', 'hidden_dim', 'batch_size', 'anneal_fraction', 'epoch', 'val_total', 'val_reconstruction', 'val_kl', 'val_reference', 'val_pixel_mse')) + ' |'
# Markdown 표를 노트북에 표시함
display(Markdown(table))
# 단계별로 무엇을 선택했는지 확인함
for selection in selections:
    # 단계명과 선택된 실험 이름을 출력함
    print(selection['stage'], '→', selection['selected_trial'])

## 3. 같은 검증 이미지의 복원 비교

각 그림의 위 행은 동일한 원본 10장, 아래 행은 `z=mu`를 사용한 복원입니다. 선명도뿐 아니라 획 누락과 숫자 모양 보존도 확인하세요. 모델 선택 점수는 잠재 샘플링으로 계산하므로 이 결정적 복원 이미지의 오차와 완전히 같지는 않습니다.

In [ ]:
# baseline과 각 단계의 선택 모델을 중복 없이 정리함
selected_trials = list(dict.fromkeys([rows[0]['trial']] + [s['selected_trial'] for s in selections]))
# 비교할 모델의 복원 이미지를 하나씩 표시함
for trial in selected_trials:
    # 저장된 원본/복원 이미지를 읽음
    image = plt.imread(RESULT_DIR / trial / 'reconstruction.png')
    # 이미지를 충분한 크기로 표시함
    plt.figure(figsize=(12, 3))
    # 이미지 배열을 그대로 출력함
    plt.imshow(image)
    # 어떤 실험인지 제목으로 표시함
    plt.title(trial)
    # 이미지 외곽의 축을 숨김
    plt.axis('off')
    # 노트북에 그림을 표시함
    plt.show()

## 4. 후보별 학습 안정성과 최종 설정

각 후보 디렉토리의 `history.csv`에는 매 epoch의 effective β, train/validation total·reconstruction·KL·BCE·pixel MSE·공통 점수가 있습니다. `curves.png`와 함께 loss 진동이나 과적합을 확인하세요. 학습률이 낮은 후보는 같은 epoch 예산에서 덜 수렴했을 수 있습니다.

`best.pt`는 후보별 최적 가중치이며 최종 후보는 `winner.json`에 기록됩니다. 이 설정은 탐색에 사용한 규모와 seed에 대한 결과입니다. 전체 학습/검증 데이터로 baseline과 선택값을 재학습하여 개선 여부를 확인해야 합니다.

In [ ]:
# 실험에 사용한 조건과 최종 후보를 출력함
print((RESULT_DIR / 'manifest.json').read_text())
# dataclass 설정 항목만 결과에서 추출함
chosen_config = {key: winner[key] for key in Config.__dataclass_fields__}
# 다른 실험이나 전체 데이터 재학습에 사용할 수 있는 설정을 표시함
print(json.dumps(chosen_config, indent=2))
# 선택된 모델의 학습 및 검증 곡선을 표시함
plt.figure(figsize=(14, 4))
# total, reconstruction, KL이 각각 분리된 그림을 읽음
plt.imshow(plt.imread(RESULT_DIR / winner['trial'] / 'curves.png'))
# 그림 외곽의 축을 숨김
plt.axis('off')
# 곡선을 노트북에 표시함
plt.show()

## 5. 전체 데이터 확인 및 최종 모델 불러오기

학습 54,000장, 검증 6,000장으로 baseline과 탐색 선택값을 각각 20 epoch 학습합니다. 목표 β에 도달한 모델 중 공통 검증 점수로 최종 모델을 고른 뒤, 테스트 10,000장을 한 번 평가합니다. 선택 후 테스트 점수를 보고 모델을 다시 바꾸지 않습니다.

아래 셀은 **학습을 다시 하지 않고** 확인 결과와 저장된 최종 가중치를 불러옵니다. 전체 데이터 확인은 다음 명령으로 실행할 수 있습니다.

```bash
python examples/vae_tuning.py --confirm-from outputs/notebooks/vae/tuning_20260929 --output outputs/notebooks/vae/new_confirmation --epochs 20
```


In [ ]:
# 전체 데이터 확인 실행 결과가 있는 폴더를 선택함
CONFIRM_DIR = PROJECT_DIR / 'outputs/notebooks/vae/confirmation_20260929'
# 확인 학습이 끝났는지 결과 파일로 확인함
if (CONFIRM_DIR / 'final.json').exists():
    # 최종 선택 설정과 별도 테스트 지표를 읽음
    final = json.loads((CONFIRM_DIR / 'final.json').read_text())
    # 전체 데이터에서 비교한 지표를 출력함
    print((CONFIRM_DIR / 'summary.csv').read_text())
    # 최종 테스트 지표를 별도로 표시함
    print('Final test:', final['test_metrics'])
    # 기존 노트북의 동일 VAE 클래스와 실행 장치를 가져옴
    from examples.vae_tuning import load_model_class, DEVICE
    # 학습 시와 동일한 클래스 정의를 읽음
    VAE, _ = load_model_class()
    # 저장한 가중치와 설정을 CPU로 불러옴
    checkpoint = torch.load(CONFIRM_DIR / 'selected.pt', map_location='cpu', weights_only=True)
    # 가중치와 함께 저장된 모델 크기를 사용함
    config = checkpoint['config']
    # 같은 잠재 차원과 hidden dimension으로 모델을 만듦
    tuned_model = VAE(config['latent_dim'], config['hidden_dim']).to(DEVICE)
    # 최종 선택한 가중치와 BatchNorm 통계를 적용함
    tuned_model.load_state_dict(checkpoint['model_state'])
    # 이후 복원과 생성을 위한 평가 모드로 설정함
    tuned_model.eval()
    # 전체 데이터에서의 baseline과 tuned 복원 이미지를 함께 확인함
    for label in ('baseline', 'tuned'):
        # 동일 검증 이미지에 대한 복원 그림을 생성함
        plt.figure(figsize=(12, 3))
        # 해당 실험에서 저장한 원본/복원 비교 그림을 읽음
        plt.imshow(plt.imread(CONFIRM_DIR / label / 'reconstruction.png'))
        # 어떤 설정의 복원인지 표시함
        plt.title('Full data: ' + label)
        # 이미지 외곽의 눈금을 숨김
        plt.axis('off')
        # 복원 비교 그림을 출력함
        plt.show()
# 전체 데이터 확인 결과가 아직 없으면 탐색 결과까지만 확인할 수 있음을 안내함
else:
    # 전체 확인 실행 방법은 위 설명에 있음
    print('전체 데이터 확인 결과가 아직 없습니다. 위 명령으로 생성할 수 있습니다.')